# Challenge 1 Data Read + Tract-Level Merge Template

This notebook is a ready-to-run starter for **Challenge 1: Demographic Travel Patterns**.

It includes:
- dataset discovery from local `data/` or Colab Google Drive data root;
- loading both CSV and spatial formats (SHP/GPKG/GeoJSON) when available;
- tract-level merge logic using tract IDs and spatial aggregation helpers.

In [ ]:
from pathlib import Path
import re
import warnings

import pandas as pd
import geopandas as gpd

from helper.data_read import (
    read_csv_file,
    read_shp_file,
    read_gpkg_file,
    read_geojson_file,
)
from helper.spatial import (
    summarize_points_by_polygon,
    summarize_lines_by_polygon,
    summarize_polygons_by_polygon,
    add_nearest_distance,
    prepare_spatial_data,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

In [ ]:
def find_repo_root(start: Path) -> Path:
    current = start.resolve()
    for path in [current] + list(current.parents):
        if (path / "helper").exists() and (path / "challenges").exists():
            return path
    return current


REPO_ROOT = find_repo_root(Path.cwd())
DATA_ROOT_CANDIDATES = [
    REPO_ROOT / "data",
    Path("/content/drive/MyDrive/data"),
    Path("/content/data"),
    Path("data"),
]
DATA_ROOT = next((p for p in DATA_ROOT_CANDIDATES if p.exists()), None)

print(f"Repo root: {REPO_ROOT}")
print(f"Data root: {DATA_ROOT}")
if DATA_ROOT is None:
    raise FileNotFoundError("No data root found. Add or mount a data folder first.")


def locate_dataset_folder(data_root: Path, folder_candidates, keyword_tokens):
    for name in folder_candidates:
        p = data_root / name
        if p.exists() and p.is_dir():
            return p

    dirs = [d for d in data_root.iterdir() if d.is_dir()]
    tokens = [t.lower() for t in keyword_tokens]
    scored = []
    for d in dirs:
        name = d.name.lower()
        score = sum(1 for t in tokens if t in name)
        if score > 0:
            scored.append((score, d))

    if not scored:
        return None

    scored.sort(key=lambda x: x[0], reverse=True)
    return scored[0][1]


def choose_preferred_csv(csv_files):
    if not csv_files:
        return None
    filtered = [
        f for f in csv_files
        if "column-metadata" not in f.name.lower() and "notes" not in f.name.lower()
    ]
    return sorted(filtered or csv_files)[0]


def first_match(folder: Path, patterns):
    for pattern in patterns:
        matches = sorted(folder.rglob(pattern))
        if matches:
            return matches[0]
    return None


def read_by_extension(path: Path):
    ext = path.suffix.lower()
    if ext == ".csv":
        return read_csv_file(str(path))
    if ext == ".shp":
        return read_shp_file(str(path))
    if ext == ".gpkg":
        return read_gpkg_file(str(path))
    if ext in {".geojson", ".json"}:
        return read_geojson_file(str(path))
    raise ValueError(f"Unsupported extension: {ext}")

In [ ]:
DATASET_SPECS = [
    {
        "id": "census",
        "folder_candidates": ["census_demographic_2024"],
        "keyword_tokens": ["census", "demographic"],
    },
    {
        "id": "replica_od",
        "folder_candidates": ["replica_od", "od", "replica"],
        "keyword_tokens": ["replica", "od", "trip"],
    },
    {
        "id": "dart_bus",
        "folder_candidates": ["dallas_dart_bus"],
        "keyword_tokens": ["dart", "bus"],
    },
    {
        "id": "aadt",
        "folder_candidates": ["annual_average_daily_traffic_dallas_county_2024"],
        "keyword_tokens": ["annual", "daily", "traffic", "aadt"],
    },
    {
        "id": "employers",
        "folder_candidates": ["employers_dallas_county"],
        "keyword_tokens": ["employer"],
    },
    {
        "id": "development",
        "folder_candidates": ["development_dallas_county"],
        "keyword_tokens": ["development"],
    },
    {
        "id": "features",
        "folder_candidates": ["features_dallas_county"],
        "keyword_tokens": ["features"],
    },
    {
        "id": "bikeway",
        "folder_candidates": ["bikeway_dallas_county"],
        "keyword_tokens": ["bikeway", "bike"],
    },
    {
        "id": "crashes",
        "folder_candidates": ["txdot_crashes", "crashes_dallas_county"],
        "keyword_tokens": ["crash"],
    },
    {
        "id": "weather",
        "folder_candidates": ["weather"],
        "keyword_tokens": ["weather", "precip", "temperature"],
    },
]

In [ ]:
loaded_assets = {}
asset_log = []

for spec in DATASET_SPECS:
    ds_id = spec["id"]
    folder = locate_dataset_folder(
        DATA_ROOT,
        spec["folder_candidates"],
        spec["keyword_tokens"],
    )

    if folder is None:
        asset_log.append({"dataset": ds_id, "status": "not found", "path": None})
        continue

    csv_file = choose_preferred_csv(sorted(folder.rglob("*.csv")))
    shp_file = first_match(folder, ["*.shp"])
    gpkg_file = first_match(folder, ["*.gpkg"])
    geojson_file = first_match(folder, ["*.geojson", "*.json"])

    for label, file_path in [
        ("csv", csv_file),
        ("shp", shp_file),
        ("gpkg", gpkg_file),
        ("geojson", geojson_file),
    ]:
        if file_path is None:
            continue

        key = f"{ds_id}_{label}"
        try:
            loaded_assets[key] = read_by_extension(file_path)
            asset_log.append(
                {
                    "dataset": ds_id,
                    "status": "loaded",
                    "format": label,
                    "path": str(file_path),
                    "rows": len(loaded_assets[key]),
                    "cols": len(loaded_assets[key].columns),
                }
            )
        except Exception as exc:
            asset_log.append(
                {
                    "dataset": ds_id,
                    "status": f"failed: {exc}",
                    "format": label,
                    "path": str(file_path),
                }
            )

asset_log_df = pd.DataFrame(asset_log)
print(f"Loaded assets: {len(loaded_assets)}")
display(asset_log_df)

## Tract-Level Merge and Spatial Aggregation

This section creates a tract-level analysis table by:
- merging tabular data using tract IDs where available;
- summarizing point, line, and polygon spatial layers to tracts;
- adding nearest-distance features for accessibility context.

In [ ]:
def find_tract_boundary_file(data_root: Path):
    geo_patterns = ["*.shp", "*.gpkg", "*.geojson", "*.json"]
    candidates = []
    for pattern in geo_patterns:
        for f in data_root.rglob(pattern):
            name = f.name.lower()
            if "tract" in name and any(k in name for k in ["census", "boundary", "2020", "tl_"]):
                candidates.append(f)
    return sorted(candidates)[0] if candidates else None


def add_tract_id(df):
    tract_cols = [
        "tract_id",
        "GEOID",
        "GEO_ID",
        "geoid",
        "origin_trct_2020",
        "destination_trct_2020",
        "TRACTCE",
    ]
    out = df.copy()
    source_col = next((c for c in tract_cols if c in out.columns), None)
    if source_col is None:
        out["tract_id"] = pd.NA
        return out

    vals = out[source_col].astype(str).str.extract(r"(\d{11})", expand=False)
    if vals.isna().all():
        vals = out[source_col].astype(str).str.extract(r"(\d{6})", expand=False)
    out["tract_id"] = vals
    return out


tract_file = find_tract_boundary_file(DATA_ROOT)
if tract_file is None:
    raise FileNotFoundError("No tract boundary spatial file found in data root.")

tracts = read_by_extension(tract_file)
tracts = add_tract_id(tracts)
tracts = prepare_spatial_data(tracts, projected_epsg=2276)
tracts = tracts[tracts["tract_id"].notna()].copy()

print(f"Tract layer: {tract_file}")
print(tracts.shape)
tracts.head(2)

In [ ]:
tract_summary = tracts.copy()

# 1) Merge tract-based tabular datasets.
for key, obj in loaded_assets.items():
    if isinstance(obj, gpd.GeoDataFrame):
        continue

    df = add_tract_id(obj)
    if "tract_id" not in df.columns or df["tract_id"].isna().all():
        continue

    numeric_cols = [c for c in df.columns if pd.api.types.is_numeric_dtype(df[c])]
    numeric_cols = [c for c in numeric_cols if c.lower() not in {"objectid"}]
    keep_cols = ["tract_id"] + numeric_cols[:20]
    grouped = df[keep_cols].groupby("tract_id", as_index=False).mean(numeric_only=True)

    rename_map = {c: f"{key}_{c}" for c in grouped.columns if c != "tract_id"}
    grouped = grouped.rename(columns=rename_map)
    tract_summary = tract_summary.merge(grouped, on="tract_id", how="left")


# 2) Spatially aggregate non-tract spatial layers.
for key, obj in loaded_assets.items():
    if not isinstance(obj, gpd.GeoDataFrame):
        continue

    gdf = obj.copy()
    try:
        gdf = prepare_spatial_data(gdf, projected_epsg=2276)
    except Exception:
        continue

    if gdf.empty:
        continue

    geom_names = set(gdf.geom_type.astype(str).str.lower().unique())

    try:
        if geom_names <= {"point", "multipoint"}:
            tract_summary = summarize_points_by_polygon(
                tract_summary,
                gdf,
                count_column=f"{key}_count",
                predicate="intersects",
            )
        elif geom_names <= {"linestring", "multilinestring"}:
            tract_summary = summarize_lines_by_polygon(
                tract_summary,
                gdf,
                length_column=f"{key}_miles",
                count_column=f"{key}_segments",
                unit="miles",
                projected_epsg=2276,
            )
        elif geom_names <= {"polygon", "multipolygon"}:
            tract_summary = summarize_polygons_by_polygon(
                tract_summary,
                gdf,
                area_column=f"{key}_acres",
                count_column=f"{key}_polygons",
                unit="acres",
                projected_epsg=2276,
            )
    except Exception as exc:
        print(f"Skipped spatial aggregation for {key}: {exc}")


# 3) Example accessibility features from tract centroids.
if "dart_bus_shp" in loaded_assets and isinstance(loaded_assets["dart_bus_shp"], gpd.GeoDataFrame):
    tract_summary = add_nearest_distance(
        tract_summary,
        prepare_spatial_data(loaded_assets["dart_bus_shp"], projected_epsg=2276),
        distance_column="dist_to_nearest_bus_stop_miles",
        unit="miles",
        projected_epsg=2276,
        source_geometry="centroid",
    )

if "employers_shp" in loaded_assets and isinstance(loaded_assets["employers_shp"], gpd.GeoDataFrame):
    tract_summary = add_nearest_distance(
        tract_summary,
        prepare_spatial_data(loaded_assets["employers_shp"], projected_epsg=2276),
        distance_column="dist_to_nearest_employer_miles",
        unit="miles",
        projected_epsg=2276,
        source_geometry="centroid",
    )

print(tract_summary.shape)
display(tract_summary.head(3))

In [ ]:
OUTPUT_DIR = REPO_ROOT / "outputs" / "challenge_1"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

tab_out = OUTPUT_DIR / "challenge_1_tract_summary.csv"
geo_out = OUTPUT_DIR / "challenge_1_tract_summary.geojson"

pd.DataFrame(tract_summary.drop(columns=tract_summary.geometry.name)).to_csv(tab_out, index=False)
tract_summary.to_file(geo_out, driver="GeoJSON")

print(f"Saved table: {tab_out}")
print(f"Saved geospatial output: {geo_out}")